# Exploration du data lake OMEGA LAKE

Lit, pour de vrai, un exemple de chaque format réellement présent dans le lake — CSV, JSON (liste), NDJSON, Parquet (`staging/` et `curated/`) — complément interactif à
[`docs/architecture/catalogue_omega_lake.md`](../docs/architecture/catalogue_omega_lake.md).

**Prérequis** : `docker compose -f infra/docker/docker-compose.yml up -d db minio minio-init api-mock`, puis avoir fait tourner au moins une fois `python3 -m datacore.storage.lake.ingestion_batch`, `python3 -m datacore.storage.lake.sse_consumer` (quelques secondes suffisent) et `python3 -m datacore.storage.lake.transform` pour peupler les 3 zones.

**Instantané** : les chemins observés ici datent du 23/09/2026 — une nouvelle exécution de l'ingestion batch crée une nouvelle partition `date=` sous `raw/` (le nombre d'objets et leurs noms évoluent avec le temps). La liste à jour s'obtient toujours avec la première cellule ci-dessous, ou `python3 -m datacore.storage.lake.catalogue`, jamais figée dans ce notebook.

In [1]:
import sys

sys.path.insert(0, "..")  # pour importer datacore si le notebook est lance depuis notebooks/

from datacore.config import OMEGA_LAKE_BUCKET
from datacore.storage.lake.ingestion_batch import client
from datacore.storage.lake.transform import connexion

s3 = client()
con = connexion()

print(f"Contenu reel du bucket {OMEGA_LAKE_BUCKET} :\n")
for zone in ("raw", "staging", "curated"):
    objets = s3.list_objects_v2(Bucket=OMEGA_LAKE_BUCKET, Prefix=f"{zone}/").get("Contents", [])
    print(f"{zone}/ ({len(objets)} objets)")
    for o in sorted(objets, key=lambda x: x["Key"]):
        print(f"  s3://{OMEGA_LAKE_BUCKET}/{o['Key']}")

Contenu reel du bucket omega-lake :

raw/ (7 objets)
  s3://omega-lake/raw/camera_comptage/date=2026-09-23/camera_comptage.csv
  s3://omega-lake/raw/capteurs_temperature/date=2026-09-23/capteurs_temperature.csv
  s3://omega-lake/raw/flux_sse_capteurs/date=2026-09-23/part-102732160283.ndjson
  s3://omega-lake/raw/flux_sse_capteurs/date=2026-09-23/part-102736163048.ndjson
  s3://omega-lake/raw/flux_sse_capteurs/date=2026-09-23/part-102738178957.ndjson
  s3://omega-lake/raw/geoloc_flotte/date=2026-09-23/geoloc_flotte.csv
  s3://omega-lake/raw/rfid_scans/date=2026-09-23/rfid_scans.json
staging/ (5 objets)
  s3://omega-lake/staging/camera_comptage/part-0.parquet
  s3://omega-lake/staging/capteurs_temperature/part-0.parquet
  s3://omega-lake/staging/flux_sse_capteurs/part-0.parquet
  s3://omega-lake/staging/geoloc_flotte/part-0.parquet
  s3://omega-lake/staging/rfid_scans/part-0.parquet
curated/ (5 objets)
  s3://omega-lake/curated/camera_comptage/part-0.parquet
  s3://omega-lake/curated/cap

## 1. Lire un CSV (`raw/`)

Format natif, copié tel quel par l'ingestion batch — aucune transformation à ce stade.

In [2]:
con.sql(f"""
    SELECT * FROM read_csv_auto('s3://{OMEGA_LAKE_BUCKET}/raw/capteurs_temperature/date=*/*.csv')
    LIMIT 5
""").show()

┌─────────────────────┬──────────┬───────────────┬───────────────┬────────┬────────────┐
│      timestamp      │ entrepot │     zone      │ temperature_c │ alerte │    date    │
│      timestamp      │ varchar  │    varchar    │    double     │ int64  │    date    │
├─────────────────────┼──────────┼───────────────┼───────────────┼────────┼────────────┤
│ 2026-08-01 00:00:00 │ OMG-LYO  │ Zone froide A │          3.38 │      0 │ 2026-09-23 │
│ 2026-08-01 00:15:00 │ OMG-LYO  │ Zone froide A │           2.1 │      0 │ 2026-09-23 │
│ 2026-08-01 00:30:00 │ OMG-LYO  │ Zone froide A │          3.14 │      0 │ 2026-09-23 │
│ 2026-08-01 00:45:00 │ OMG-LYO  │ Zone froide A │          2.18 │      0 │ 2026-09-23 │
│ 2026-08-01 01:00:00 │ OMG-LYO  │ Zone froide A │          2.79 │      0 │ 2026-09-23 │
└─────────────────────┴──────────┴───────────────┴───────────────┴────────┴────────────┘



## 2. Lire un JSON liste (`raw/rfid_scans`)

Fichier `[{...}, {...}, ...]` — `read_json_auto` sans option particulière suffit.

In [3]:
con.sql(f"""
    SELECT * FROM read_json_auto('s3://{OMEGA_LAKE_BUCKET}/raw/rfid_scans/date=*/*.json')
    LIMIT 5
""").show()

┌─────────┬─────────────────────┬────────────┬──────────┬────────────┬─────────────┬────────────┐
│ scan_id │      timestamp      │ palette_id │ entrepot │    zone    │ produit_sku │    date    │
│  int64  │      timestamp      │  varchar   │ varchar  │  varchar   │   varchar   │    date    │
├─────────┼─────────────────────┼────────────┼──────────┼────────────┼─────────────┼────────────┤
│       1 │ 2026-08-03 18:31:00 │ PAL-72981  │ OMG-MAR  │ Expedition │ SKU-30025   │ 2026-09-23 │
│       2 │ 2026-08-03 05:26:00 │ PAL-66949  │ OMG-MAR  │ Reception  │ SKU-10006   │ 2026-09-23 │
│       3 │ 2026-08-02 23:19:00 │ PAL-61767  │ OMG-MAR  │ Stockage   │ SKU-20011   │ 2026-09-23 │
│       4 │ 2026-08-01 21:53:00 │ PAL-40108  │ OMG-LYO  │ Expedition │ SKU-30030   │ 2026-09-23 │
│       5 │ 2026-08-02 12:46:00 │ PAL-99271  │ OMG-MAR  │ Reception  │ SKU-30021   │ 2026-09-23 │
└─────────┴─────────────────────┴────────────┴──────────┴────────────┴─────────────┴────────────┘



## 3. Lire un NDJSON (`raw/flux_sse_capteurs`)

Un objet JSON par ligne, déposé par le consommateur SSE — nécessite `format='newline_delimited'`, sans quoi DuckDB tente de lire tout le fichier comme un seul document JSON et échoue.

In [4]:
con.sql(f"""
    SELECT * FROM read_json_auto(
        's3://{OMEGA_LAKE_BUCKET}/raw/flux_sse_capteurs/date=*/*.ndjson',
        format='newline_delimited'
    )
""").show()

┌────────────────────────────┬──────────┬───────────────┬───────────────┬─────────────┬──────────┬─────────┬────────────┐
│         timestamp          │ entrepot │     zone      │ temperature_c │ vehicule_id │   lat    │   lon   │    date    │
│          varchar           │ varchar  │    varchar    │    double     │   varchar   │  double  │ double  │    date    │
├────────────────────────────┼──────────┼───────────────┼───────────────┼─────────────┼──────────┼─────────┼────────────┤
│ 2026-09-23T08:27:28.154342 │ OMG-MAR  │ Zone froide A │          3.51 │ VH-001      │ 42.91293 │ 2.61811 │ 2026-09-23 │
│ 2026-09-23T08:27:30.157769 │ OMG-LYO  │ Zone froide A │           3.2 │ VH-010      │ 44.90435 │ 2.61484 │ 2026-09-23 │
│ 2026-09-23T08:27:32.159215 │ OMG-LIL  │ Zone froide B │          2.83 │ VH-008      │ 46.43577 │ 5.86389 │ 2026-09-23 │
│ 2026-09-23T08:27:34.161137 │ OMG-LYO  │ Zone froide A │          4.89 │ VH-015      │ 46.81647 │ 5.75815 │ 2026-09-23 │
│ 2026-09-23T08:27:36.16

## 4. Lire un Parquet `staging/` (typé, sans jointure)

`geoloc_flotte` : pas de jointure en `curated/` non plus (décision RGPD, C18 §6) — `staging/` et `curated/` sont donc identiques pour ce flux.

In [5]:
con.sql(f"""
    SELECT * FROM read_parquet('s3://{OMEGA_LAKE_BUCKET}/staging/geoloc_flotte/part-0.parquet')
    LIMIT 5
""").show()

┌─────────────────────┬─────────────┬──────────┬──────────┬─────────────┬────────────┐
│      timestamp      │ vehicule_id │   lat    │   lon    │ vitesse_kmh │    date    │
│      timestamp      │   varchar   │  double  │  double  │    int64    │    date    │
├─────────────────────┼─────────────┼──────────┼──────────┼─────────────┼────────────┤
│ 2026-08-01 02:00:00 │ VH-001      │ 46.95629 │ -1.40059 │          50 │ 2026-09-23 │
│ 2026-08-01 08:00:00 │ VH-001      │ 47.12373 │ -1.34987 │           0 │ 2026-09-23 │
│ 2026-08-01 09:30:00 │ VH-001      │ 47.39862 │ -1.27408 │          50 │ 2026-09-23 │
│ 2026-08-01 19:30:00 │ VH-001      │  47.2226 │ -1.58954 │          70 │ 2026-09-23 │
│ 2026-08-02 02:30:00 │ VH-001      │ 47.44376 │ -1.68228 │          50 │ 2026-09-23 │
└─────────────────────┴─────────────┴──────────┴──────────┴─────────────┴────────────┘



## 5. Lire un Parquet `curated/` (jointure visible)

`rfid_scans` : `entrepot_nom`, `produit_libelle`, `temperature_dirigee` viennent de la jointure à `dimensions.dim_site`/`dimensions.dim_produit` (entrepôt OMEGA BI) — absents du `staging/` équivalent.

In [6]:
con.sql(f"""
    SELECT * FROM read_parquet('s3://{OMEGA_LAKE_BUCKET}/curated/rfid_scans/part-0.parquet')
    LIMIT 5
""").show()

┌─────────┬─────────────────────┬────────────┬──────────┬────────────┬─────────────┬────────────┬──────────────────────────┬──────────────────────┬─────────────────────┐
│ scan_id │      timestamp      │ palette_id │ entrepot │    zone    │ produit_sku │    date    │       entrepot_nom       │   produit_libelle    │ temperature_dirigee │
│  int64  │      timestamp      │  varchar   │ varchar  │  varchar   │   varchar   │    date    │         varchar          │       varchar        │       boolean       │
├─────────┼─────────────────────┼────────────┼──────────┼────────────┼─────────────┼────────────┼──────────────────────────┼──────────────────────┼─────────────────────┤
│      28 │ 2026-08-03 00:41:00 │ PAL-98789  │ OMG-MAR  │ Reception  │ SKU-30022   │ 2026-09-23 │ Entrepot Omega Marseille │ Jean coupe droite    │ false               │
│      33 │ 2026-08-02 06:18:00 │ PAL-47991  │ OMG-LIL  │ Reception  │ SKU-30025   │ 2026-09-23 │ Entrepot Omega Lille     │ Pull col rond        │ fa

## Aller plus loin

- Inventaire complet et à jour (source/zone/format/schéma/fraîcheur) : `python3 -m datacore.storage.lake.catalogue`.
- Une requête ponctuelle hors notebook, en une ligne de terminal (`.venv` actif) :

```bash
python3 -c "
from datacore.storage.lake.transform import connexion
con = connexion()
con.sql(\"SELECT * FROM 's3://omega-lake/curated/rfid_scans/part-0.parquet' LIMIT 20\").show()
"
```

`connexion()` fait tout le travail de configuration (extensions `httpfs`/`postgres`, endpoint MinIO, style d'adressage `path`) — inutile de le retaper à la main à chaque fois.